# 🎬 Reels Autopilot: مجاني 100٪

**رابط فيديو ← Gemini يختار أفضل اللحظات ← مقاطع عمودية بترجمة متحركة وتكبير خفيف ← عنوان ووصف وكلمات مفتاحية وهاشتاقات لكل منصة ← Google Drive وGoogle Sheets ← نشر**

### طريقة الاستخدام
1. **أول مرة فقط:** `Runtime ← Change runtime type ← T4 GPU`، وضع `GEMINI_API_KEY` في 🔑 Secrets.
2. شغّل المربع **1** ثم المربع **2**.
3. في المربع **3** الصق الرابط، واختر الإعدادات، ثم اضغط ▶️.

In [ ]:
# @title 1) ⚙️ التثبيت (دقيقة واحدة)
!apt-get -qq install -y ffmpeg fonts-dejavu-core > /dev/null
!rm -rf /content/app && git clone -q --depth 1 -b claude/auto-reels-shorts-multiplatform-cy6p5z https://github.com/bluesky100090099/free-claude-code /content/app
%cd /content/app/reels_autopilot
!pip -q install -r requirements.txt
!pip -q install -U yt-dlp
print("✅ تم التثبيت")

In [ ]:
# @title 2) 🔑 تحميل المفاتيح
import os

from google.colab import userdata

for k in ["GEMINI_API_KEY", "YT_CLIENT_ID", "YT_CLIENT_SECRET", "YT_REFRESH_TOKEN",
          "IG_USER_ID", "IG_ACCESS_TOKEN", "FB_PAGE_ID", "FB_PAGE_TOKEN",
          "TIKTOK_CLIENT_KEY", "TIKTOK_CLIENT_SECRET", "TIKTOK_REFRESH_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k) or ""
    except Exception:
        pass
!python -m reels doctor

In [ ]:
# @title 3) ⚡ الصق الرابط واضغط ▶️
# @markdown **الروابط:** رابط واحد أو أكثر، تفصل بينها مسافة. تقبل روابط YouTube وTikTok وInstagram وFacebook وGoogle Drive وDropbox، أو أي رابط مباشر لملف mp4.
LINKS = ""  # @param {type:"string"}
# @markdown أو فعّل الخيار التالي لترفع فيديو من هاتفك بدلاً من الرابط:
UPLOAD_FROM_PHONE = False  # @param {type:"boolean"}
# @markdown ---
CLIPS = 5  # @param {type:"slider", min:1, max:15, step:1}
SECONDS = 15  # @param {type:"slider", min:5, max:60, step:1}
LANGUAGE = "ar"  # @param ["ar", "en", "fr", "es", "tr", "de"]
FRAMING = "smart"  # @param ["smart", "blur", "center"]
ZOOM_EFFECT = True  # @param {type:"boolean"}
CAPTIONS = True  # @param {type:"boolean"}
# @markdown ---
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
LOG_TO_SHEET = True  # @param {type:"boolean"}
PUBLISH = "لا، سأنشر يدوياً"  # @param ["لا، سأنشر يدوياً", "مقطع واحد الآن", "كل المقاطع الآن"]

import subprocess
import sys

inputs = LINKS.replace(",", " ").split()
if UPLOAD_FROM_PHONE:
    from google.colab import files
    inputs += [os.path.abspath(n) for n in files.upload()]
if not inputs:
    raise SystemExit("⚠️ ضع رابطاً في خانة LINKS أو فعّل UPLOAD_FROM_PHONE")

if SAVE_TO_DRIVE or LOG_TO_SHEET:
    from google.colab import auth
    print("🔐 وافق على نافذة جوجل حتى نتمكن من الحفظ في Drive وSheets…")
    auth.authenticate_user()

cmd = [sys.executable, "-u", "-m", "reels", "make", *inputs, "-n", str(CLIPS),
       "-d", str(SECONDS), "--lang", LANGUAGE, "--crop", FRAMING, "-o", "/content/out",
       "--zoom", "0.06" if ZOOM_EFFECT else "0"]
if not CAPTIONS:
    cmd.append("--no-captions")
if SAVE_TO_DRIVE:
    cmd.append("--drive")
if LOG_TO_SHEET:
    cmd.append("--sheet")
if PUBLISH != "لا، سأنشر يدوياً":
    cmd += ["--publish", "--next", "1" if PUBLISH == "مقطع واحد الآن" else "0"]

BATCHES = []
print("⏳ بدأ العمل… قد يستغرق بضع دقائق لكل فيديو. أبقِ الصفحة مفتوحة.\n")
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")
        if line.startswith("BATCH_DIR="):
            BATCHES.append(line.strip().split("=", 1)[1])
print("\n🎉 انتهى! شغّل المربع 4 لمشاهدة المقاطع." if BATCHES else "\n❌ لم تُصنع مقاطع. أرسل لقطة شاشة لما فوق.")

In [ ]:
# @title 4) 👀 مشاهدة المقاطع والنصوص
import base64
import json

from IPython.display import HTML, display

for BATCH in BATCHES:
    with open(f"{BATCH}/manifest.json") as fh:
        m = json.load(fh)
    html = "<div style='display:flex;flex-wrap:wrap;gap:16px'>"
    for c in m["clips"]:
        with open(f"{BATCH}/{c['file']}", "rb") as fh:
            b64 = base64.b64encode(fh.read()).decode()
        html += (
            "<div style='width:260px'>"
            f"<video width=260 controls src='data:video/mp4;base64,{b64}'></video>"
            f"<p dir='auto'><b>{c['title']}</b><br>{c['caption']}<br>"
            f"<span style='color:#1a73e8'>{' '.join(c['hashtags'])}</span></p></div>"
        )
    display(HTML(html + "</div>"))
print("📋 النصوص الجاهزة لكل منصة موجودة في ملف POST_KIT.md داخل مجلد Drive، وفي Google Sheet.")

In [ ]:
# @title 5) 💾 تنزيل كل المقاطع إلى الهاتف (ZIP)
import shutil

from google.colab import files

for BATCH in BATCHES:
    files.download(shutil.make_archive(f"/content/{os.path.basename(BATCH)}", "zip", BATCH))